# Bölüm 12: Modelinizi Eğitmek

> "Başarısız olmadım. Sadece işe yaramayacak 10.000 yol buldum."
> — **Thomas Edison**, Mucit

---

## Neler Öğreneceksiniz

- Sonraki token tahminin modellere yazmayı nasıl öğrettiği
- Tüm sinir ağlarına güç veren 5 adımlık eğitim tarifi
- Kayıp eğrilerinin modelinizin öğrenip öğrenmediğini nasıl ortaya çıkardığı
- Aşırı öğrenmeyi nasıl önleyeceğiniz ve ne zaman durmanız gerektiği
- İlerlemenizi asla kaybetmemek için kontrol noktaları kaydetme
- Modelin Shakespeare metninin ritmini ve satır yapısını nasıl öğrendiğini görmek

---

## Kurulum

İlk olarak, gerekli paketleri yükleyelim:

In [ ]:
# Ek A’da test edilen sürümleri yükle
!pip install -q torch==2.15.0 transformers==5.23.0 tqdm==4.70.1 numpy==2.3.5 matplotlib==3.10.7

In [ ]:
# ===== KÜTÜPHANELER =====
import math
import urllib.request
from functools import partial
from dataclasses import dataclass

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import AutoTokenizer, get_linear_schedule_with_warmup
from tqdm import tqdm

# Cihazı ayarla
device = torch.device("cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu"))
print(f"Kullanılan cihaz: {device}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# ===== YENİDEN ÜRETİLEBİLİRLİK =====
import random
import numpy as np

def set_seed(seed=42):
    """Kullandığımız her rastgele kaynağı tohumla."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    if torch.backends.mps.is_available():
        torch.mps.manual_seed(seed)

set_seed(42)

## 1. Bölüm 10-11'den Model Bileşenleri

İlk olarak, önceki bölümlerde oluşturduğumuz MiniGPT modelini getirelim.

In [ ]:
# ===== ÇOK BAŞLI DİKKAT (Bölüm 10'dan) =====
class MultiHeadAttention(nn.Module):
    """
    Verimli çok başlı dikkat (tüm başları birlikte yığınlar).
    """
    def __init__(self, d_model, num_heads, dropout=0.1):
        super().__init__()
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"

        self.d_model = d_model
        self.num_heads = num_heads
        self.d_head = d_model // num_heads  # 768 / 12 = 64

        # Birleşik QKV projeksiyonu (ayrı ayrı yapmaktan 3 kat verimli!)
        # Neden 3 * d_model? Çünkü Q, K, V'ye aynı anda projeksiyon yapıyoruz
        # bias=True GPT-2 ile eşleşir; böylece önceden eğitilmiş bias'larını sonra yükleyebiliriz
        self.qkv_proj = nn.Linear(d_model, 3 * d_model)

        # Çıktı projeksiyonu
        self.out_proj = nn.Linear(d_model, d_model)

        # Dikkat ağırlıklarında dropout
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        batch, seq, d_model = x.shape

        # ===== Adım 1: Q, K, V'ye projeksiyon (hepsi tek seferde!) =====
        qkv = self.qkv_proj(x)  # (batch, seq, 3 * d_model)

        # ===== Adım 2: Q, K, V'ye böl ve çok başlı için yeniden şekillendir =====
        # (batch, seq, 3, num_heads, d_head) şekline yeniden şekillendir
        qkv = qkv.reshape(batch, seq, 3, self.num_heads, self.d_head)

        # (3, batch, num_heads, seq, d_head) düzenine geçir
        qkv = qkv.permute(2, 0, 3, 1, 4)

        # Q, K, V'ye böl: her biri (batch, num_heads, seq, d_head)
        Q, K, V = qkv[0], qkv[1], qkv[2]

        # ===== Adım 3: Ölçeklendirilmiş nokta çarpım dikkati (başlar üzerinde yığınlanmış) =====
        d_k = self.d_head
        scores = Q @ K.transpose(-2, -1)  # (batch, num_heads, seq, seq)
        scores = scores / math.sqrt(d_k)

        # Verildiyse nedensel maskeyi uygula
        if mask is not None:
            # Maskeyi başlar için genişlet: (seq, seq) → (1, 1, seq, seq)
            if mask.dim() == 2:
                mask = mask.unsqueeze(0).unsqueeze(0)
            scores = scores.masked_fill(mask == 0, float('-inf'))

        attn_weights = F.softmax(scores, dim=-1)  # (batch, num_heads, seq, seq)
        attn_weights = self.dropout(attn_weights)

        # Değerlerin ağırlıklı toplamı
        attn_output = attn_weights @ V  # (batch, num_heads, seq, d_head)

        # ===== Adım 4: Başları birleştir =====
        # (batch, seq, num_heads, d_head) düzenine transpoz et
        attn_output = attn_output.transpose(1, 2)

        # (batch, seq, d_model) şekline yeniden şekillendir; bu başları birleştirir
        attn_output = attn_output.reshape(batch, seq, d_model)

        # ===== Adım 5: Son projeksiyon =====
        output = self.out_proj(attn_output)

        return output, attn_weights


# Test et

print("MultiHeadAttention tanımlandı!")

In [ ]:
# ===== İLERİ BESLEMELİ AĞ (Bölüm 10'dan) =====
class FeedForward(nn.Module):
    """
    Konuma göre ileri beslemeli ağ.
    Her konuma bağımsız olarak uygulanır (tüm konumlar için aynı ağırlıklar).
    """
    def __init__(self, d_model, d_ff, dropout=0.1):
        """
        Argümanlar:
            d_model: Model boyutu (GPT-2 small için 768)
            d_ff: İleri beslemeli boyutu (tipik olarak 4 * d_model = 3072)
            dropout: Dropout olasılığı
        """
        super().__init__()
        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        # x: (batch, seq, d_model)
        x = self.fc1(x)        # (batch, seq, d_ff), genişlet
        x = F.gelu(x, approximate="tanh")  # Doğrusal olmayan (GPT-2'nin GELU varyantı)
        x = self.dropout(x)
        x = self.fc2(x)        # (batch, seq, d_model), geri projeksiyon
        return x


# Test et

print("FeedForward tanımlandı!")

In [ ]:
# ===== TRANSFORMER BLOĞU (Bölüm 10'dan) =====
class TransformerBlock(nn.Module):
    """
    Çok başlı dikkat, ileri beslemeli, artık bağlantılar ve
    katman normalizasyonu ile tam Transformer bloğu (GPT-2 gibi ön-norm stili).
    """
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        """
        Argümanlar:
            d_model: Model boyutu (768)
            num_heads: Dikkat başı sayısı (12)
            d_ff: İleri beslemeli boyutu (3072 = 4 * d_model)
            dropout: Dropout olasılığı
        """
        super().__init__()

        # Katman normalizasyonu (her alt katmandan önce)
        self.ln1 = nn.LayerNorm(d_model)
        self.ln2 = nn.LayerNorm(d_model)

        # Çok başlı dikkat
        self.attn = MultiHeadAttention(d_model, num_heads, dropout)

        # İleri beslemeli ağ
        self.ffn = FeedForward(d_model, d_ff, dropout)

        # Dropout (her alt katmandan sonra uygulanır)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        """
        Argümanlar:
            x: Girdi gömmeleri (batch, seq, d_model)
            mask: Nedensel maske (seq, seq) veya (1, 1, seq, seq)

        Döndürür:
            x: Çıktı gömmeleri (batch, seq, d_model)
            attn_weights: Dikkat ağırlıkları (batch, heads, seq, seq)
        """
        # ===== Artık Bağlantılı Çok Başlı Dikkat =====
        # Ön-norm: Dikkatten ÖNCE normalleştir
        attn_out, attn_weights = self.attn(self.ln1(x), mask)
        x = x + self.dropout(attn_out)  # Artık bağlantı

        # ===== Artık Bağlantılı İleri Besleme =====
        # Ön-norm: FFN'den ÖNCE normalleştir
        ffn_out = self.ffn(self.ln2(x))
        x = x + self.dropout(ffn_out)  # Artık bağlantı

        return x, attn_weights


# Tam bir Transformer bloğunu test et

print("TransformerBlock tanımlandı!")

In [ ]:
# ===== GPT YAPILANDIRMASI (Bölüm 11'den) =====
from dataclasses import dataclass

@dataclass
class GPTConfig:
    """MiniGPT modeli için yapılandırma."""
    vocab_size: int = 50257      # GPT-2 kelime dağarcığı boyutu
    max_seq_len: int = 1024      # Maksimum bağlam uzunluğu
    embed_dim: int = 768         # Gömme boyutu (d_model)
    num_heads: int = 12          # Dikkat başı sayısı
    num_layers: int = 12         # Transformer bloğu sayısı
    d_ff: int = 3072             # İleri beslemeli gizli boyut (4 * embed_dim)
    dropout: float = 0.1         # Dropout olasılığı

    def __post_init__(self):
        """Başlatmadan sonra yapılandırmayı doğrula."""
        assert self.embed_dim % self.num_heads == 0, \
            f"embed_dim ({self.embed_dim}), num_heads " \
            f"({self.num_heads}) ile bölünebilir olmalıdır"


In [ ]:
# ===== MİNİGPT MODELİ (Bölüm 11'den) =====
class MiniGPT(nn.Module):
    """
    Minimal bir GPT tarzı dil modeli.
    Gömmeleri, Transformer bloklarını ve dil modelleme başını birleştirir.
    """

    def __init__(self, config: GPTConfig):
        super().__init__()
        self.config = config

        # ===== Token ve Konum Gömmeleri (Bölüm 9) =====
        self.token_embed = nn.Embedding(config.vocab_size, config.embed_dim)
        self.pos_embed = nn.Embedding(config.max_seq_len, config.embed_dim)
        self.dropout = nn.Dropout(config.dropout)

        # ===== Transformer Blokları (Bölüm 10) =====
        self.blocks = nn.ModuleList([
            TransformerBlock(
                d_model=config.embed_dim,
                num_heads=config.num_heads,
                d_ff=config.d_ff,
                dropout=config.dropout
            )
            for _ in range(config.num_layers)
        ])

        # ===== Son LayerNorm =====
        self.ln_f = nn.LayerNorm(config.embed_dim)

        # ===== Dil Modelleme Başı =====
        self.lm_head = nn.Linear(config.embed_dim, config.vocab_size, bias=False)

        # ===== Ağırlık Bağlama =====
        # Gömme ve çıktı projeksiyonu arasında ağırlıkları paylaş
        self.lm_head.weight = self.token_embed.weight

        # Initialize weights
        self._init_weights()

    def _init_weights(self):
        """Ağırlıkları küçük rastgele değerlerle başlat."""
        nn.init.normal_(self.token_embed.weight, std=0.02)
        nn.init.normal_(self.pos_embed.weight, std=0.02)

    def forward(self, token_ids, return_attention=False):
        """
        Model üzerinden ileri geçiş.

        Argümanlar:
            token_ids: Girdi token ID'leri (batch, seq)
            return_attention: Dikkat ağırlıklarının döndürülüp döndürülmeyeceği

        Döndürür:
            logits: Kelime dağarcığı skorları (batch, seq, vocab_size)
            attention_weights: Blok başına dikkat ağırlıklarının isteğe bağlı listesi
        """
        batch, seq = token_ids.shape
        device = token_ids.device

        # ===== Gömmeler =====
        # Token gömmeleri
        tok_emb = self.token_embed(token_ids)  # (batch, seq, embed_dim)

        # Konum gömmeleri
        positions = torch.arange(seq, device=device)  # (seq,)
        pos_emb = self.pos_embed(positions)  # (seq, embed_dim)

        # Birleştir ve dropout uygula
        x = self.dropout(tok_emb + pos_emb)  # (batch, seq, embed_dim)

        # ===== Nedensel Maske Oluştur =====
        # Alt üçgen: her konum yalnızca önceki konumlara dikkat edebilir
        mask = torch.tril(torch.ones(seq, seq, device=device))

        # ===== Transformer Bloklarından Geçir =====
        attention_weights = []
        for block in self.blocks:
            x, attn = block(x, mask)
            if return_attention:
                attention_weights.append(attn)

        # ===== Son LayerNorm =====
        x = self.ln_f(x)  # (batch, seq, embed_dim)

        # ===== Dil Modelleme Başı =====
        logits = self.lm_head(x)  # (batch, seq, vocab_size)

        if return_attention:
            return logits, attention_weights
        return logits

    @torch.no_grad()
    def generate(self, input_ids, max_new_tokens=50, temperature=1.0,
                 do_sample=True, eos_token_id=None):
        """
        input_ids'i devam ettiren token'lar üret (istem çıktıya dahildir).

        Argümanlar:
            input_ids: İstem token ID'leri (batch, seq)
            max_new_tokens: Kaç yeni token üretilecek
            temperature: Örnekleme sıcaklığı (yalnızca do_sample=True iken kullanılır)
            do_sample: Dağılımdan örnekle; False = açgözlü argmax
            eos_token_id: Her dizi bu token'ı üretirse erken dur

        Döndürür:
            Token ID'leri (batch, seq + yeni token'lar)
        """
        self.eval()
        for _ in range(max_new_tokens):
            logits = self(input_ids)
            next_logits = logits[:, -1, :]  # (batch, vocab_size)

            if do_sample:
                probs = F.softmax(next_logits / temperature, dim=-1)
                next_token = torch.multinomial(probs, num_samples=1)
            else:
                next_token = next_logits.argmax(dim=-1, keepdim=True)

            input_ids = torch.cat([input_ids, next_token], dim=1)

            if eos_token_id is not None and (next_token == eos_token_id).all():
                break

        return input_ids


## 2. Veri Kümesini İndirme

TinyShakespeare kullanacağız, dakikalar içinde eğitmek için yeterince küçük.

In [ ]:
# TinyShakespeare'i indir
url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
urllib.request.urlretrieve(url, "shakespeare.txt")

with open("shakespeare.txt", "r") as f:
    text = f.read()

print(f"Veri kümesi boyutu: {len(text):,} karakter")
print(f"\nÖrnek:\n{text[:500]}")

## 3. Veri Kümesi ve DataLoader Oluşturma

`create_lm_labels`'ın ne yapmadığına dikkat: hiçbir şeyi maskelemez. Dolguları eğitim ve değerlendirme döngülerinde dikkat maskesiyle maskeleriz (`targets[attention_mask[:, 1:] == 0] = -100`); böylece gerçek bir EOS denetimli kalır. GPT-2, EOS token'ı ile doldurur (ID 50256): değere dayalı bir maske dolguyu gerçek metin sonundan ayırt edemez ve her gerçek EOS tahminini sessizce silerek modele hiç durmamayı öğretirdi.


In [ ]:
def create_lm_labels(token_ids):
    """
    Dil modelleme için kaydırılmış girdiler ve hedefler oluştur.

    Argümanlar:
        token_ids: Token ID'leri (batch, seq)

    Döndürür:
        inputs: (batch, seq-1), son hariç tüm token'lar
        targets: (batch, seq-1), ilk hariç tüm token'lar
    """
    inputs = token_ids[:, :-1]    # Son token hariç her şey
    targets = token_ids[:, 1:].clone()  # .clone(): döngüler dolguları yerinde -100 işaretler

    # Burada maskeleme yok: dolgular döngülerde dikkat maskesiyle maskelenir;
    # böylece gerçek bir EOS (GPT-2 eos ile doldurur, ID 50256) denetimli kalır.
    return inputs, targets


from torch.utils.data import Dataset, DataLoader

class TextDataset(Dataset):
    """Metin parçaları döndüren basit veri kümesi."""

    def __init__(self, text, chunk_size=512):
        """
        Argümanlar:
            text: Metnin tamamı, bir string olarak
            chunk_size: Parça başına kaç karakter
        """
        self.chunks = []

        # Son parça tam doluysa onu da al; yalnızca eksik kalanı at.
        for i in range(0, len(text) - chunk_size + 1, chunk_size):
            self.chunks.append(text[i:i + chunk_size])

    def __len__(self):
        return len(self.chunks)

    def __getitem__(self, idx):
        return self.chunks[idx]


# Veri kümesi oluştur


def collate_fn(batch, tokenizer, max_length=128):
    """
    Bir yığın metin dizisini tokenize et ve doldur.

    Argümanlar:
        batch: Metin dizilerinin listesi
        tokenizer: Kullanılacak tokenizer
        max_length: Maksimum dizi uzunluğu

    Döndürür:
        input_ids: Doldurulmuş token ID'leri (batch, max_length)
        attention_mask: gerçek token'lar için 1, dolgu için 0
    """
    encoded = tokenizer(
        batch,
        padding=True,
        truncation=True,
        max_length=max_length,
        return_tensors="pt"
    )
    return encoded["input_ids"], encoded["attention_mask"]


# Tokenizer'ı yükle (GPT-2'nin varsayılan pad token'ı yoktur; metin sonunu yeniden kullan)
tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

# Veri kümesi oluştur: TinyShakespeare'de 512 karakterlik parçalar ortalamada
# ~156 GPT-2 token'ıdır; max_length=128 kesmesi çoğu parçanın kuyruğunu atar.
# Ön eğitim için sorun değil: her parça hâlâ yoğun eğitim sinyalidir
dataset = TextDataset(text, chunk_size=512)
print(f"Parça sayısı: {len(dataset)}")

In [ ]:
# Aynı eserin parçalarını ayırıyoruz: doğrulama, yeni belgelerde genellemeyi değil,
# bu derlemedeki başarımı ölçer. Belge düzeyinde ayırmak daha sıkı bir sınamadır.
train_size = int(0.9 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = torch.utils.data.random_split(
    dataset, [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
)

print(f"Eğitim: {len(train_dataset)}, Doğrulama: {len(val_dataset)}")

# DataLoader'ları oluştur
collate = partial(collate_fn, tokenizer=tokenizer, max_length=128)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    collate_fn=collate
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    collate_fn=collate
)

# Bir yığını kontrol et
input_ids, attention_mask = next(iter(train_loader))
print(f"Yığın input_ids şekli: {input_ids.shape}")

## 4. Model Oluşturma

In [ ]:
# Hızlı eğitim için küçük yapılandırma
config = GPTConfig(
    vocab_size=50257,
    max_seq_len=128,
    embed_dim=256,
    num_heads=4,
    num_layers=4,
    d_ff=1024,
    dropout=0.1
)

model = MiniGPT(config).to(device)
print(f"Parametreler: {sum(p.numel() for p in model.parameters()):,}")

## 5. "Önce" Durumu: Eğitilmemiş Model

In [ ]:
@torch.no_grad()
def generate(model, tokenizer, prompt, max_new_tokens=50, temperature=1.0):
    """
    Sıcaklık kontrolüyle metin üret.

    Sıcaklık:
    - < 1.0: Daha odaklı/deterministik
    - = 1.0: Orijinal dağılım
    - > 1.0: Daha rastgele/yaratıcı

    Argümanlar:
        model: Eğitilmiş model
        tokenizer: Tokenizer
        prompt: Başlangıç metni
        max_new_tokens: Kaç token üretilecek
        temperature: Örnekleme sıcaklığı

    Döndürür:
        Üretilen metin dizisi
    """
    device = next(model.parameters()).device
    token_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)

    # do_sample=True: logit'leri sıcaklığa böl, sonra örnekle
    output_ids = model.generate(
        token_ids,
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        do_sample=True,
        eos_token_id=tokenizer.eos_token_id,
    )

    return tokenizer.decode(output_ids[0])


# Eğitilmemiş modelden üret
print("EĞİTİMDEN ÖNCE (rastgele ağırlıklar):")
print("="*50)
prompt = "The king"
print(f"İstem: '{prompt}'")
print(f"Çıktı: {generate(model, tokenizer, prompt)}")
print("\n(Rastgele saçmalık, model henüz hiçbir şey öğrenmedi!)")

## 6. Eğitim Kurulumu

In [ ]:
# Eğitim hiperparametreleri
num_epochs = 3
learning_rate = 3e-4
warmup_steps = 100

total_steps = len(train_loader) * num_epochs
print(f"Toplam eğitim adımı: {total_steps}")

# Optimize edici ve zamanlayıcı
# Bölüm 11'den parametre grupları yardımcısı
def get_parameter_groups(model, weight_decay=0.1):
    decay_params = []
    no_decay_params = []
    for name, param in model.named_parameters():
        if param.ndim < 2:
            no_decay_params.append(param)
        else:
            decay_params.append(param)
    return [
        {'params': decay_params, 'weight_decay': weight_decay},
        {'params': no_decay_params, 'weight_decay': 0.0}
    ]

# Bölüm 11'den parametre grupları yardımcısı: matrisler ağırlık azaltması alır,
# bias ve LayerNorm vektörleri almaz
optimizer = AdamW(get_parameter_groups(model, weight_decay=0.01),
                  lr=learning_rate)
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)

## 7. Eğitim ve Değerlendirme Fonksiyonları

In [ ]:
def train_epoch(model, dataloader, optimizer, scheduler, device, clip_norm=1.0):
    """
    Bir dönem için eğit.

    Döndürür:
        Dönem boyunca ortalama kayıp
    """
    model.train()  # Dropout'u etkinleştir
    total_loss = 0

    progress = tqdm(dataloader, desc="Eğitim")
    for input_ids, attention_mask in progress:
        # Cihaza taşı
        input_ids = input_ids.to(device)
        attention_mask = attention_mask.to(device)

        # Girdileri ve hedefleri oluştur (bir kaydırılmış), dolgular -100 ile işaretli
        inputs, targets = create_lm_labels(input_ids)
        targets[attention_mask[:, 1:] == 0] = -100  # dolguları kayıpta göz ardı et

        # ===== 5 ADIMLI TARİF =====

        # Adım 1: Gradyanları sıfırla
        optimizer.zero_grad(set_to_none=True)

        # Adım 2: İleri geçiş
        logits = model(inputs)

        # Adım 3: Kaybı hesapla
        loss = F.cross_entropy(
            logits.view(-1, logits.size(-1)),
            targets.reshape(-1),  # reshape, bitişik olmayan dilimleri işler
            ignore_index=-100  # Dolgu konumlarını atla
        )

        # Adım 4: Geri geçiş
        loss.backward()

        # Önemli: backward()'dan SONRA ama step()'ten ÖNCE kırp
        torch.nn.utils.clip_grad_norm_(model.parameters(), clip_norm)

        # Adım 5: Ağırlıkları güncelle
        optimizer.step()
        scheduler.step()

        # ===== 5 ADIMLI TARİFİN SONU =====

        total_loss += loss.item()

        # İlerleme çubuğunu güncelle
        progress.set_postfix(loss=f"{loss.item():.4f}")

    return total_loss / len(dataloader)


@torch.no_grad()
def evaluate(model, dataloader, device):
    """
    Modeli bir veri kümesi üzerinde değerlendir.

    Döndürür:
        avg_loss: Ortalama çapraz entropi kaybı
        perplexity: Üstel alınmış kayıp (yorumlanabilir metrik)
    """
    model.eval()  # Dropout'u kapat
    total_loss = 0
    total_tokens = 0

    for input_ids, attention_mask in dataloader:
        input_ids = input_ids.to(device)
        attention_mask = attention_mask.to(device)

        inputs, targets = create_lm_labels(input_ids)
        targets[attention_mask[:, 1:] == 0] = -100  # dolguları kayıpta göz ardı et

        logits = model(inputs)

        # Kaybı hesapla
        loss = F.cross_entropy(
            logits.view(-1, logits.size(-1)),
            targets.reshape(-1),  # reshape, bitişik olmayan dilimleri işler
            ignore_index=-100,
            reduction='sum'  # Ortalama yerine toplam
        )

        # Yalnızca dolgu olmayan token'ları say; böylece perplexity dürüst olur
        total_loss += loss.item()
        total_tokens += (targets != -100).sum().item()

    avg_loss = total_loss / total_tokens
    perplexity = math.exp(avg_loss)

    return avg_loss, perplexity


def should_stop_early(val_losses, patience=3):
    """
    Eğitimi durdurup durdurmamamız gerektiğini kontrol et.

    Argümanlar:
        val_losses: Dönem başına doğrulama kayıplarının listesi
        patience: Kaç artan döneme tahammül edilecek

    Döndürür:
        Durdurmamız gerekiyorsa True
    """
    if len(val_losses) < patience + 1:
        return False

    # Son 'patience' kaybın hepsi artıyor mu kontrol et
    recent = val_losses[-(patience + 1):]
    for i in range(1, len(recent)):
        if recent[i] <= recent[i-1]:
            return False

    return True


In [ ]:
def save_checkpoint(model, optimizer, scheduler, epoch, train_loss, val_loss, path):
    """
    Eğitim kontrol noktasını kaydet (tam devam için RNG durumları dahil).

    Argümanlar:
        model: Model
        optimizer: Optimizatör (momentum durumu var)
        scheduler: Öğrenme oranı zamanlayıcısı
        epoch: Geçerli dönem numarası
        train_loss: Bu dönemin eğitim kaybı
        val_loss: Bu dönemin doğrulama kaybı
        path: Nereye kaydedilecek
    """
    torch.save({
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'train_loss': train_loss,
        'val_loss': val_loss,
        # RNG durumları: dropout ve karıştırma aynı diziyi sürdürmeli
        'rng_python': random.getstate(),
        'rng_numpy': np.random.get_state(),
        'rng_torch': torch.get_rng_state(),
        'rng_cuda': torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None,
        'rng_mps': torch.mps.get_rng_state() if torch.backends.mps.is_available() else None,
    }, path)
    print(f"Kontrol noktası kaydedildi: {path}")


def load_checkpoint(model, optimizer, scheduler, path, device):
    """
    Eğitim kontrol noktasını yükle ve RNG durumlarını geri yükle.

    Döndürür:
        epoch: Devam edilecek dönem
        train_loss: Son eğitim kaybı
        val_loss: Son doğrulama kaybı
    """
    checkpoint = torch.load(path, map_location=device, weights_only=False)

    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    scheduler.load_state_dict(checkpoint['scheduler_state_dict'])

    # RNG durumları CPU tensörü olmalı: map_location=device onları taşıyabilir
    random.setstate(checkpoint['rng_python'])
    np.random.set_state(checkpoint['rng_numpy'])
    torch.set_rng_state(checkpoint['rng_torch'].cpu())
    if checkpoint['rng_cuda'] is not None and torch.cuda.is_available():
        torch.cuda.set_rng_state_all([s.cpu() for s in checkpoint['rng_cuda']])
    if checkpoint['rng_mps'] is not None and torch.backends.mps.is_available():
        torch.mps.set_rng_state(checkpoint['rng_mps'].cpu())

    print(f"Kontrol noktası {checkpoint['epoch']}. dönemden yüklendi")

    return checkpoint['epoch'], checkpoint['train_loss'], checkpoint['val_loss']


Belirtilmesi gereken bir ödünleşim: `weights_only=True`, yüklemede keyfi pickle kodu çalıştırmayı reddeder; güvenilmeyen kaynaklardan gelen kontrol noktaları için doğru alışkanlıktır. Ama NumPy'nin RNG durumu, güvenli yükleyicinin reddettiği NumPy skalerleri içerir; bu yüzden tam devam eden bu sürüm `weights_only=False` kullanır. Bu şekilde yalnızca kendi oluşturduğunuz (veya açıkça güvendiğiniz) kontrol noktalarını yükleyin; indirilen her şey için `weights_only=True` tercih edin ve RNG durumlarının geri yüklenmeyeceğini kabul edin. Bit-bit aynı devam için iki koşul daha: zamanlayıcıyı özgün çalışmayla aynı toplam adım ufkuyla yeniden kurun ve yığınları aynı sırada tüketin.

## 8. Eğitim Döngüsü

In [ ]:
# Eğitim!
train_losses = []
val_losses = []
best_val_loss = float('inf')

for epoch in range(num_epochs):
    print(f"\n{'='*50}")
    print(f"Epoch {epoch + 1}/{num_epochs}")
    print(f"{'='*50}")

    # Eğit
    train_loss = train_epoch(model, train_loader, optimizer, scheduler, device)
    train_losses.append(train_loss)

    # Değerlendir
    val_loss, perplexity = evaluate(model, val_loader, device)
    val_losses.append(val_loss)

    print(f"\nEğitim Kaybı: {train_loss:.4f}")
    print(f"Doğrulama Kaybı:   {val_loss:.4f}")
    print(f"Perpleksite: {perplexity:.1f}")

    # En iyi kontrol noktasını kaydet
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        save_checkpoint(model, optimizer, scheduler, epoch,
                       train_loss, val_loss, "best_model.pt")

print("\nEğitim tamamlandı!")
print(f"En iyi doğrulama kaybı: {best_val_loss:.4f}")

## 9. Kayıp Eğrilerini Çizme

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))
epochs = range(1, len(train_losses) + 1)
plt.plot(epochs, train_losses, 'b-o', label='Eğitim Kaybı')
plt.plot(epochs, val_losses, 'r-o', label='Doğrulama Kaybı')
plt.xlabel('Epoch')
plt.ylabel('Kayıp')
plt.title('Eğitim ve Doğrulama Kaybı')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 10. Ödül: Metin Üretimi!

In [ ]:
# En iyi modeli yükle (kendi kontrol noktamız; weights_only=False burada güvenli,
# içindeki RNG durumları tam yükleyici ister; kontrol noktası bölümüne bakın)
checkpoint = torch.load("best_model.pt", map_location=device, weights_only=False)
model.load_state_dict(checkpoint['model_state_dict'])
print(f"Kontrol noktası {checkpoint['epoch'] + 1}. epoch'tan yüklendi")
print(f"Doğrulama kaybı: {checkpoint['val_loss']:.4f}")

In [ ]:
print("\n" + "="*60)
print("EĞİTİMDEN SONRA (Shakespeare üzerinde):")
print("="*60)

prompts = [
    "The king",
    "To be or not to be",
    "Friends, Romans, countrymen",
    "All the world's a stage"
]

for prompt in prompts:
    output = generate(model, tokenizer, prompt, max_new_tokens=40, temperature=0.8)
    print(f"\nİstem: '{prompt}'")
    print(f"Çıktı: {output}")
    print("-"*40)

## 11. Önce ve Sonra Karşılaştırması

In [ ]:
# Karşılaştırma için yeni eğitilmemiş model oluştur
set_seed(123)  # Farklı rastgele ağırlıklar için farklı tohum
untrained_model = MiniGPT(config).to(device)
untrained_model.eval()

prompt = "The fair maiden"

print("="*60)
print(f"İstem: '{prompt}'")
print("="*60)
print("\nÖNCE (eğitilmemiş):")
print(generate(untrained_model, tokenizer, prompt, max_new_tokens=30))
print("\nSONRA (Shakespeare üzerinde eğitilmiş):")
print(generate(model, tokenizer, prompt, max_new_tokens=30, temperature=0.8))
print("\nAynı mimari. Aynı kod. Eğitim tüm farkı yaratıyor!")

## 12. Sıcaklık Karşılaştırması

In [ ]:
prompt = "The noble lord"

print(f"İstem: '{prompt}'\n")
for temp in [0.5, 0.8, 1.0, 1.5]:
    output = generate(model, tokenizer, prompt, max_new_tokens=30, temperature=temp)
    print(f"Sıcaklık {temp}:")
    print(f"  {output}")
    print()

## Özet

**Neler oluşturduk:**

1. Sonraki token tahmini için **etiket kaydırma**
2. Metni verimli bir şekilde yığınlayan ve tokenize eden **DataLoader'lar**
3. **5 adımlık eğitim tarifi**: zero_grad → forward → loss → backward → step
4. Doğrulama kaybı ve perpleksite ile **değerlendirme**
5. Eğitimi kaydetmek ve devam ettirmek için **kontrol noktası oluşturma**
6. Sıcaklık kontrolü ile **metin üretimi**

**Temel kavramlar:**

- Eğitim bir geri besleme kontrol döngüsüdür: hatayı ölç, ağırlıkları ayarla, tekrarla
- Çapraz entropi kaybı, belirsiz olanlardan daha fazla kendinden emin yanlış tahminleri cezalandırır
- Aşırı öğrenme = eğitim verisini ezberlemek (eğitim kaybı ↓, doğrulama kaybı ↑)
- Perpleksite, modelin "ne kadar şaşırdığını" ölçer, düşük olanı daha iyidir
- Sıcaklık, üretim çeşitliliğini kontrol eder

**Sonrası:** Bölüm 13'te bu modelin yanıtlarını SSS verisiyle LoRA ince ayarından önce ve sonra karşılaştıracaksınız. Bölüm 15'teki uygulamalar ise MiniGPT yerine API üzerinden hazır bir talimat modeli kullanır.

## Alıştırmalar

### Alıştırma 1: Öğrenme Oranı Deneyi

Farklı öğrenme oranlarını deneyin ve sonuçları karşılaştırın.

In [ ]:
# KODUNUZ BURAYA
# lr=1e-5, lr=3e-4, lr=1e-2 ile eğitin
# Kayıp eğrilerini karşılaştırın
# Hangi öğrenme oranı en iyi çalışıyor?

### Alıştırma 2: Daha Fazla Epoch

In [ ]:
# KODUNUZ BURAYA
# 3 yerine 5-10 epoch için eğitin
# Model gelişmeye devam ediyor mu?
# Aşırı öğrenme belirtileri görüyor musunuz?

### Alıştırma 3: Model Boyutu Karşılaştırması

In [ ]:
# KODUNUZ BURAYA
# Daha küçük bir model oluşturun (2 katman, 128 boyut)
# Daha büyük bir model oluşturun (6 katman, 384 boyut)
# Eğitim hızını ve nihai perpleksiteyi karşılaştırın